# Day 08 · 하네스로 개발한다

7강까지는 RAG·MCP·에이전트 루프를 **손으로** 짰다. 오늘은 **코드 에이전트에게 다시 만들게 하고, 내가 판정한다.**

> 내가 짜봤기 때문에 판정할 수 있다. 에이전트 시대에 제일 어려운 능력이 바로 **출력을 판정하는 것**이다.

모든 랩은 3단이다 — **입힌다 → 돌려본다 → 원리**.

| 랩 | 입히는 것 | 과제 |
|---|---|---|
| Lab 1 | (없음) 맨몸 | 5강 MCP 서버 다시 만들기 |
| Lab 2 | 스킬 | 2강 RAG 파이프라인 다시 만들기 |
| Lab 3 | 플러그인 | 규약을 팀에 배포 |
| Lab 4a | 훅 | 정책 강제 |
| Lab 4b | MCP 등록 | **에이전트가 만든 도구를 에이전트가 쓴다** |
| Lab 5 | (파이썬) | 미니 하네스 직접 — 하네스 내부를 손으로 |

**준비** — Claude Code 설치·로그인이 필요하다 (Pro/Max/Team/Console 플랜. 무료 플랜은 불가). Codex는 선택. 설치는 `day08/Claude_Code_Codex_세팅.md` 참고.


## Lab 1 · 맨몸 하네스로 MCP 서버 만들기

**입힌다** — 아무것도 안 얹는다. 순수 Claude Code.

빈 폴더를 만들고 `claude` 를 켠 뒤 그대로 시킨다:

```
5강에서 만든 것과 같은 MCP 서버를 만들어줘.
FastMCP 로 도구 3개: search_docs(읽기), add_task(추가), set_priority(검증 있는 변경).
set_priority 는 '낮음·보통·높음'만 허용하고, 아니면 친절한 에러를 낸다.
```

**돌려본다** — 관찰 항목:
- [ ] 계획을 먼저 세우나, 바로 파일부터 쓰나
- [ ] 어떤 도구를 부르나 (Read · Write · Bash …)
- [ ] 파일을 쓰기 전에 **승인을 묻나**
- [ ] 다 만들고 **스스로 실행해서 확인하나**

**원리** — 하네스도 결국 **판단 · 도구 · 루프**다. 7강에서 짠 `run_agent` 와 뼈대가 같다. 다른 건 규모 — 계획·컨텍스트·권한이 그 위에 얹혔을 뿐이다.

**판정** — 내가 짠 5강 서버와 비교한다. 무엇이 빠졌나? `inputSchema` 는 제대로 나오나? 에러 메시지는 **모델이 읽고 스스로 고칠 만한가**?


## Lab 2 · 스킬을 얹으면 출력이 달라진다

**입힌다** — 우리 규약을 스킬로 만든다. `writing-skills` 스킬이 있으면 시켜서 만들고, 없으면 직접 쓴다.

`.claude/skills/our-rules/SKILL.md`:

```markdown
---
name: our-rules
description: 우리 프로젝트 규약. 파이썬 코드를 쓰기 전에 읽는다.
---

- 임베딩은 sentence-transformers 를 쓴다
- 검색 함수는 (query, k) 시그니처를 지킨다
- 하드코딩된 API 키를 절대 쓰지 않는다 (환경변수·입력창)
- 만든 코드는 반드시 실행해서 확인한다
```

**돌려본다** — 같은 요청을 **스킬 없이 / 스킬 얹고** 각각 돌려 비교한다:

```
2강에서 만든 것 같은 RAG 파이프라인을 만들어줘.
문서를 청킹하고, 임베딩해서, 질문과 가장 가까운 k개를 찾아 돌려준다.
```

**원리** — 스킬 = **필요할 때 로드되는 노하우**. 모델을 안 바꾸고 출력을 바꾼다. 생성된 SKILL.md 를 **열어본다** — 프론트매터 + 마크다운, 그게 전부다.

**판정** — 내가 짠 2강 RAG와 비교. 청킹 전략은? 정규화는? 벡터 검색의 한계를 아는가?


## Lab 3 · 플러그인으로 묶어 배포

**입힌다** — 스킬을 플러그인으로 묶는다. 매니페스트는 `.claude-plugin/plugin.json` (필수).

**돌려본다** — 다른 폴더에서 설치해 **같은 규약이 따라오는지** 확인한다.

**원리** — 플러그인 = 확장의 **배포 단위**. 스킬·훅·서브에이전트·MCP를 한 덩어리로 묶어 팀에 뿌린다.


## Lab 4a · 훅으로 정책 강제

**입힌다** — PreToolUse 훅. 에이전트가 도구를 부르기 **전에** 내 스크립트가 통과/거부를 결정한다 (exit 2 = 거부).

**돌려본다** — "테스트 없이 커밋 금지" 같은 규칙을 걸고, 에이전트가 어기려 할 때 **막히는지** 본다.

**원리** — **스킬은 부탁, 훅은 강제.** 안전은 부탁으로 만들어지지 않는다.


## Lab 4b · 만든 MCP를 자기가 쓴다 (한 바퀴 닫기)

**입힌다** — Lab 1에서 **에이전트가 만든 MCP 서버**를 하네스에 등록한다.

```powershell
claude mcp add day08-tools -- python C:\경로\server.py    # 절대경로 권장
claude mcp list
```

Codex라면:

```powershell
codex mcp add day08-tools -- python server.py
```

**돌려본다** — `claude` 를 켜고 `/mcp` 로 연결을 확인한 뒤, 자연어로 시킨다:

```
day08-tools 로 '분기 보고서' 할 일을 추가하고 우선순위를 긴급으로 설정해줘
```

'긴급'은 허용값이 아니다 → 도구가 친절한 에러를 낸다 → **하네스가 스스로 '높음'으로 고쳐 재시도하는지** 본다.

**원리** — 도구가 없으면 **만들어 붙인다**. 에이전트가 만든 도구를 에이전트가 쓴다 — 메타 루프가 여기서 닫힌다. 7강 Lab 4의 self-correction이 하네스 규모에서 그대로 재현된다.


## Lab 5 · 미니 하네스 직접

하네스의 네 조각을 직접 조립한다: **도구 레지스트리 + 에이전트 루프(7강) + 컨텍스트 트리밍 + 확장점(스킬)**. (NVIDIA Qwen 필요)

In [ ]:
%pip install -q openai

In [ ]:
# Lab 5(미니 하네스)에서 LLM으로 Qwen을 부른다. 토큰은 입력창/환경변수 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank", "thinking"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("모델:", LLM_MODEL)

In [ ]:
# Lab 5 — 미니 하네스 = 도구 레지스트리 + 에이전트 루프 + 컨텍스트 트리밍 + 확장점(스킬)
class MiniHarness:
    """'미니 Claude Code' — 하네스의 네 조각을 직접 조립한다."""
    def __init__(self, system, keep=10):
        self.system = system
        self.registry = {}      # 이름 → 함수
        self.specs = []         # OpenAI 도구 스펙
        self.skills = {}        # 확장점: 트리거 단어 → 노하우
        self.keep = keep        # 컨텍스트로 남길 최근 메시지 수

    def tool(self, description, params):                 # 도구 등록 데코레이터
        def deco(fn):
            self.registry[fn.__name__] = fn
            self.specs.append({"type": "function", "function": {
                "name": fn.__name__, "description": description,
                "parameters": {"type": "object",
                               "properties": {k: {"type": v} for k, v in params.items()},
                               "required": list(params)}}})
            return fn
        return deco

    def skill(self, trigger, text):                      # 확장점: 트리거 단어가 나오면 노하우 주입
        self.skills[trigger] = text

    def _system(self, q):
        extra = [t for k, t in self.skills.items() if k in q]
        return self.system + ("\n\n[스킬]\n" + "\n".join(extra) if extra else "")

    def _trim(self, messages):                           # 컨텍스트 트리밍: system + 최근 keep개
        if len(messages) <= self.keep + 1:
            return messages
        return messages[:1] + messages[-self.keep:]

    def run(self, question, max_steps=6):
        messages = [{"role": "system", "content": self._system(question)},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=self._trim(messages), tools=self.specs,
                temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  [도구] {tc.function.name}({args})")
                try:
                    out = self.registry[tc.function.name](**args)
                except Exception as e:
                    out = f"오류: {e}"
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"

In [ ]:
# 미니 하네스에 도구 2개를 등록하고, 스킬(확장점)을 하나 붙여 돌려본다
harness = MiniHarness("너는 도구를 쓰는 조수다. 한국어로 간결히 답하라.")

@harness.tool(description="두 정수를 더한다", params={"a": "integer", "b": "integer"})
def add(a, b):
    return a + b

@harness.tool(description="도시의 (예시) 날씨를 반환한다", params={"city": "string"})
def weather(city):
    return {"city": city, "temp": 21, "sky": "맑음"}

harness.skill("날씨", "날씨를 물으면 weather 도구를 쓰고 섭씨로 답한다.")   # 확장점(스킬) 주입

print("A:", harness.run("서울 날씨 알려주고, 3 더하기 4도 계산해줘"))

## Lab · (Claude Code) 하네스 프레임워크 체험

스킬·플러그인의 실제 사례 — 유명 프레임워크를 설치해 기본 Claude Code와 비교한다. **로컬 Claude Code 터미널**에서 진행(노트북 커널 아님).

```bash
# Superpowers — TDD·서브에이전트 개발 methodology (가장 인기 플러그인)
/plugin marketplace add obra/superpowers
/plugin install superpowers
```

**해볼 것**
- 같은 작업을 기본 Claude Code vs 프레임워크로 시켜보고 흐름(서브에이전트·리뷰·역할) 차이를 관찰
- 토큰 절감 스킬 `ponytail`(YAGNI·꼭 필요한 코드만)·`caveman`(출력 압축)을 붙여 실제 절감을 확인

> ⚠️ caveman은 광고(65퍼센트)와 실측(~8.5퍼센트, 출력 토큰만)이 다르다 — **직접 재보는 것**이 실습의 핵심.

## 산출물 체크리스트

- [ ] 맨몸 하네스로 **MCP 서버를 다시 만들었다** (Lab 1)
- [ ] 내가 5강에서 짠 것과 **비교해 무엇이 다른지 적었다**
- [ ] 스킬을 얹어 **같은 요청의 출력이 달라지는 걸 봤다** (Lab 2)
- [ ] 생성된 **SKILL.md 를 열어봤다** — 프론트매터 + 마크다운
- [ ] 플러그인으로 묶어 **다른 폴더에 배포했다** (Lab 3)
- [ ] 훅으로 **정책을 강제했다** (Lab 4a) — 스킬은 부탁, 훅은 강제
- [ ] **에이전트가 만든 MCP 를 등록해 에이전트가 썼다** (Lab 4b)
- [ ] 파이썬으로 **미니 하네스를 직접 만들었다** (Lab 5)
- [ ] RAG · MCP · 에이전트 루프를 **에이전트가 만든 것과 내가 짠 것으로 비교 회고했다**

> 한 줄: **7강까지는 내가 만든다. 8강부터는 에이전트가 만들고 내가 판정한다.**
